# Weighted ASP Formula Baseline

This notebook estimates the directly usable pricing formula:

```text
Predicted Price = w30 × ASP30 + w90 × ASP90 + wf × ASPf
```

`ASP30` and `ASP90` are the product's trailing 30-day and 90-day unit-weighted ASP. `ASPf` is the product family's trailing 90-day unit-weighted ASP. All rolling values exclude the current day. One global set of weights is estimated for CA and another for US; weights are not fitted separately by family.

Constraints: all weights are nonnegative and sum to one.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy.optimize import minimize

ROOT = Path.cwd()
if not (ROOT / 'Top 30 Product for US-CA').exists():
    ROOT = Path('baseline model_Brynn')
DATA_DIR = ROOT / 'Top 30 Product for US-CA'
ARTIFACT_DIR = ROOT / 'artifacts'
ARTIFACT_DIR.mkdir(exist_ok=True)
FILES = {c: DATA_DIR / f'{c}_selected_top30_transactions.csv' for c in ['CA', 'US']}

## Feature construction

For any trailing window, ASP is computed as total historical revenue divided by total historical units. Current-day transactions are excluded with `closed='left'`, preventing target leakage.

In [2]:
def rolling_asp(daily, keys, window_days, output_name):
    outputs = []
    for _, group in daily.groupby(keys, dropna=False, sort=False):
        group = group.sort_values('pricing_date').set_index('pricing_date')
        revenue = group['revenue'].rolling(f'{window_days}D', closed='left').sum()
        units = group['order_units'].rolling(f'{window_days}D', closed='left').sum()
        result = group.reset_index()[keys + ['pricing_date']].copy()
        result[output_name] = (revenue / units).to_numpy()
        outputs.append(result)
    return pd.concat(outputs, ignore_index=True)

def prepare_data(path):
    df = pd.read_csv(path, low_memory=False)
    df['pricing_date'] = pd.to_datetime(df['pricing_date'], errors='coerce')
    for column in ['asp', 'revenue', 'order_units']:
        df[column] = pd.to_numeric(df[column], errors='coerce')
    df = df.loc[
        df['pricing_date'].notna() & df['asp'].notna() & (df['asp'] > 0) &
        df['order_units'].notna() & (df['order_units'] > 0)
    ].copy()
    df['product_family'] = df['product_family'].astype('string').fillna('<missing>')

    product_daily = df.groupby(['product_id', 'pricing_date'], as_index=False)[['revenue', 'order_units']].sum()
    family_daily = df.groupby(['product_family', 'pricing_date'], as_index=False)[['revenue', 'order_units']].sum()
    asp30 = rolling_asp(product_daily, ['product_id'], 30, 'ASP30')
    asp90 = rolling_asp(product_daily, ['product_id'], 90, 'ASP90')
    aspf = rolling_asp(family_daily, ['product_family'], 90, 'ASPf')

    df = df.merge(asp30, on=['product_id', 'pricing_date'], how='left')
    df = df.merge(asp90, on=['product_id', 'pricing_date'], how='left')
    df = df.merge(aspf, on=['product_family', 'pricing_date'], how='left')
    df = df.dropna(subset=['ASP30', 'ASP90', 'ASPf']).sort_values('pricing_date').reset_index(drop=True)
    df['sample_weight'] = np.sqrt(df['order_units']).clip(1, 5)

    dates = np.array(sorted(df['pricing_date'].unique()))
    train_end = pd.Timestamp(dates[int(len(dates) * 0.70) - 1])
    validation_end = pd.Timestamp(dates[int(len(dates) * 0.85) - 1])
    train = df[df['pricing_date'] <= train_end].copy()
    validation = df[(df['pricing_date'] > train_end) & (df['pricing_date'] <= validation_end)].copy()
    test = df[df['pricing_date'] > validation_end].copy()
    return train, validation, test

splits = {country: prepare_data(path) for country, path in FILES.items()}
pd.DataFrame([
    {'country': country, 'split': name, 'rows': len(part),
     'start': part.pricing_date.min().date(), 'end': part.pricing_date.max().date()}
    for country, parts in splits.items()
    for name, part in zip(['train', 'validation', 'test'], parts)
])

,country,split,rows,start,end
0,CA,train,8300,2023-06-21,2025-10-11
1,CA,validation,4105,2025-10-12,2026-03-06
2,CA,test,4738,2026-03-07,2026-08-29
3,US,train,36217,2023-04-05,2025-08-12
4,US,validation,26045,2025-08-13,2026-02-06
5,US,test,26169,2026-02-07,2026-08-29


## Objective-function comparison

MAE, MSE, and Huber training objectives are compared. The variant with the lowest validation weighted MAE is selected. After selection, its weights are refitted on train plus validation, and the untouched test period is evaluated once.

In [3]:
FEATURES = ['ASP30', 'ASP90', 'ASPf']

def fit_weights(data, loss_name):
    X = data[FEATURES].to_numpy()
    y = data['asp'].to_numpy()
    sample_weight = data['sample_weight'].to_numpy()

    def objective(weights):
        error = y - X @ weights
        if loss_name == 'MAE':
            loss = np.abs(error)
        elif loss_name == 'MSE':
            loss = error ** 2
        else:
            delta = 100.0
            absolute_error = np.abs(error)
            loss = np.where(
                absolute_error <= delta,
                0.5 * error ** 2,
                delta * (absolute_error - 0.5 * delta),
            )
        return np.average(loss, weights=sample_weight)

    result = minimize(
        objective,
        x0=np.ones(3) / 3,
        method='SLSQP',
        bounds=[(0, 1)] * 3,
        constraints={'type': 'eq', 'fun': lambda weights: weights.sum() - 1},
        options={'maxiter': 1000, 'ftol': 1e-12},
    )
    if not result.success:
        raise RuntimeError(result.message)
    return result.x

def evaluate(data, weights):
    y = data['asp'].to_numpy()
    prediction = data[FEATURES].to_numpy() @ weights
    sample_weight = data['sample_weight'].to_numpy()
    return {
        'weighted_MAE': np.average(np.abs(y - prediction), weights=sample_weight),
        'weighted_RMSE': np.sqrt(np.average((y - prediction) ** 2, weights=sample_weight)),
        'weighted_WAPE': np.sum(sample_weight * np.abs(y - prediction)) / np.sum(sample_weight * np.abs(y)),
        'R2': 1 - np.sum(sample_weight * (y - prediction) ** 2) /
                    np.sum(sample_weight * (y - np.average(y, weights=sample_weight)) ** 2),
    }

validation_rows = []
for country, (train, validation, test) in splits.items():
    for loss_name in ['MAE', 'MSE', 'Huber']:
        weights = fit_weights(train, loss_name)
        validation_rows.append({
            'country': country, 'training_loss': loss_name,
            'w30': weights[0], 'w90': weights[1], 'wf': weights[2],
            **evaluate(validation, weights),
        })

validation_results = pd.DataFrame(validation_rows).sort_values(['country', 'weighted_MAE'])
validation_results.to_csv(ARTIFACT_DIR / 'formula_validation_comparison.csv', index=False)
validation_results

,country,training_loss,w30,w90,wf,weighted_MAE,weighted_RMSE,weighted_WAPE,R2
1,CA,MSE,0.803391,7.780196e-08,1.966092e-01,164.881772,273.830438,0.143639,0.812637
2,CA,Huber,0.862964,1.044373e-08,1.370358e-01,164.977880,276.063107,0.143722,0.809570
0,CA,MAE,0.883366,2.130731e-12,1.166335e-01,165.220015,277.106570,0.143933,0.808127
5,US,Huber,0.899582,1.004185e-01,1.381154e-15,169.405978,299.997963,0.133125,0.836180
3,US,MAE,0.882429,1.175715e-01,1.893562e-17,169.456230,299.911672,0.133165,0.836275
4,US,MSE,0.737468,2.557656e-01,6.766526e-03,170.471320,299.617711,0.133962,0.836595


In [4]:
final_rows = []
prediction_outputs = []
for country, (train, validation, test) in splits.items():
    selected_loss = validation_results.query('country == @country').iloc[0]['training_loss']
    train_validation = pd.concat([train, validation], ignore_index=True)
    weights = fit_weights(train_validation, selected_loss)
    metrics = evaluate(test, weights)
    final_rows.append({
        'country': country, 'selected_loss': selected_loss,
        'w30': weights[0], 'w90': weights[1], 'wf': weights[2],
        **metrics,
    })
    output = test[['pricing_date', 'product_id', 'product_family', 'asp', 'ASP30', 'ASP90', 'ASPf', 'sample_weight']].copy()
    output['predicted_price'] = output[FEATURES].to_numpy() @ weights
    output['country'] = country
    prediction_outputs.append(output)

final_results = pd.DataFrame(final_rows)
formula_predictions = pd.concat(prediction_outputs, ignore_index=True)
final_results.to_csv(ARTIFACT_DIR / 'formula_final_weights_and_metrics.csv', index=False)
formula_predictions.to_csv(ARTIFACT_DIR / 'formula_test_predictions.csv', index=False)
final_results

,country,selected_loss,w30,w90,wf,weighted_MAE,weighted_RMSE,weighted_WAPE,R2
0,CA,MSE,0.767614,2.592892e-16,0.232386,175.497681,344.533373,0.134517,0.620214
1,US,Huber,0.913042,7.948494e-02,0.007473,178.239055,331.883523,0.135211,0.725056


## Final formula

The next cell prints the production-style formula for each country. Coefficients are rounded only for display; exported CSV files retain full precision.

In [5]:
for row in final_results.itertuples():
    print(
        f'{row.country}: Predicted Price = '
        f'{row.w30:.4f} × ASP30 + {row.w90:.4f} × ASP90 + {row.wf:.4f} × ASPf'
    )

CA: Predicted Price = 0.7676 × ASP30 + 0.0000 × ASP90 + 0.2324 × ASPf
US: Predicted Price = 0.9130 × ASP30 + 0.0795 × ASP90 + 0.0075 × ASPf


## Interpretation and limitations

- The weights are predictive blending coefficients, not causal price-elasticity estimates.
- A zero or near-zero coefficient means that the corresponding signal did not add validation value after accounting for the other highly correlated rolling averages.
- The 30-day and 90-day ASP values can be strongly correlated, so coefficients may change when the time period changes. Monitor and refit them regularly.
- Rows without sufficient prior price history are excluded from weight estimation. A production implementation should define a cold-start fallback.